In [2]:
import os
import cv2
import numpy as np
from skimage import exposure
from tqdm import tqdm

# Paths
input_dir = "/kaggle/input/pneumonia/PNEUMONIA"  # Path to original images
output_dir = "/kaggle/working/chest_xray/augmented"  # Path to save augmented images
os.makedirs(output_dir, exist_ok=True)

# Function to add Gaussian noise
def add_gaussian_noise(image, mean=0, sigma=10):
    noise = np.random.normal(mean, sigma, image.shape).astype(np.uint8)
    noisy_image = cv2.add(image, noise)
    return noisy_image

# Function to adjust brightness & contrast
def adjust_brightness_contrast(image, alpha=1.1, beta=10):
    return cv2.convertScaleAbs(image, alpha=alpha, beta=beta)

# Function for CLAHE (Contrast Limited Adaptive Histogram Equalization)
def apply_clahe(image, clip_limit=2.0, tile_grid_size=(8, 8)):
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    return clahe.apply(image)

# Function for histogram matching using a reference image
def match_histogram(image, reference):
    return exposure.match_histograms(image, reference, channel_axis=None)


# Load a reference image for histogram matching
reference_img = cv2.imread(os.path.join(input_dir, os.listdir(input_dir)[0]), cv2.IMREAD_GRAYSCALE)

# Process images
num_augmentations = 4700  # Target total images
original_images = os.listdir(input_dir)

print(f"Generating {num_augmentations} augmented images...")

for i in tqdm(range(num_augmentations)):
    img_name = original_images[i % len(original_images)]  # Cycle through original images
    img_path = os.path.join(input_dir, img_name)
    
    image = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    
    # Randomly apply augmentations
    if np.random.rand() > 0.5:
        image = add_gaussian_noise(image)
    
    if np.random.rand() > 0.5:
        image = adjust_brightness_contrast(image, alpha=np.random.uniform(0.9, 1.2), beta=np.random.randint(-15, 15))
    
    if np.random.rand() > 0.5:
        image = apply_clahe(image)
    
    if np.random.rand() > 0.5:
        image = match_histogram(image, reference_img)

    # Save the augmented image
    save_path = os.path.join(output_dir, f"aug_{i+1}.png")
    cv2.imwrite(save_path, image)

print(f"Augmentation complete! Images saved to {output_dir}")


Generating 4700 augmented images...


100%|██████████| 4700/4700 [02:52<00:00, 27.18it/s]

Augmentation complete! Images saved to /kaggle/working/chest_xray/augmented
